<a href="https://colab.research.google.com/github/AzisilhamK463/TA-Azis_Ilham_K/blob/main/Hybrid_Learning_LSTM_%26_Autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.layers import RepeatVector
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Memuat dataset (sesuaikan dengan path dataset Anda)
file_path = '/path_to_dataset.csv'  # Ganti dengan path file yang benar
dataset = pd.read_csv(file_path)

# Menampilkan beberapa baris pertama dari dataset
print(dataset.head())

In [ ]:
# Pembuatan fitur baru untuk menangkap pola serangan
dataset['src_to_dst_bytes_ratio'] = dataset['src_bytes'] / (dataset['dst_bytes'] + 1)
dataset['packets_per_minute'] = dataset['src_pkts'] / (dataset['duration'] + 1)
dataset['duration_to_dst_bytes_ratio'] = dataset['duration'] / (dataset['dst_bytes'] + 1)
dataset['src_ip_port_count'] = dataset.groupby(['src_ip', 'src_port'])['src_ip'].transform('count')
dataset['inter_arrival_time'] = dataset['ts'].diff()
dataset['time_since_last_attack'] = dataset['ts'] - dataset['ts'].shift(1)
dataset['dns_query_count'] = dataset.groupby('src_ip')['dns_query'].transform('count')
dataset['is_dns_query'] = dataset['dns_query'].apply(lambda x: 1 if x != '-' else 0)

# Menghapus kolom yang tidak diperlukan
dataset = dataset.drop(columns=['weird_name', 'weird_addl', 'weird_notice'])

# Menampilkan dataset setelah feature engineering
print(dataset.head())

In [ ]:
# Label Encoding untuk kolom 'proto' (karena ini kolom kategorikal)
label_encoder_proto = LabelEncoder()
dataset['proto'] = label_encoder_proto.fit_transform(dataset['proto'])

# One-Hot Encoding untuk kolom 'service' dan 'http_method'
dataset = pd.get_dummies(dataset, columns=['service', 'http_method'])

# Menampilkan dataset setelah encoding
print(dataset.head())

In [ ]:
# Memilih fitur numerik untuk model
features = ['src_to_dst_bytes_ratio', 'packets_per_minute', 'duration_to_dst_bytes_ratio',
            'src_ip_port_count', 'inter_arrival_time', 'time_since_last_attack',
            'dns_query_count', 'is_dns_query', 'proto']  # Fitur yang sudah ditambahkan dan di-encode

# Menyiapkan fitur (X) dan target (y)
X = dataset[features].values
y = dataset['type'].values  # Menggunakan kolom 'type' sebagai target untuk LSTM

# Label encoding untuk kolom 'type' (mengonversi label ke dalam format numerik)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Membagi data menjadi train dan test
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Normalisasi fitur
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Mengubah data menjadi format 3D (samples, timesteps, features) untuk LSTM
X_train_scaled = np.reshape(X_train_scaled, (X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_scaled = np.reshape(X_test_scaled, (X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

In [ ]:
# Membangun model LSTM untuk klasifikasi serangan
lstm_model = Sequential()
lstm_model.add(LSTM(units=50, return_sequences=True, input_shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])))
lstm_model.add(Dropout(0.2))
lstm_model.add(LSTM(units=50, return_sequences=False))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(units=6, activation='softmax'))  # Output 6 jenis serangan (menggunakan softmax untuk multi-class classification)

# Kompilasi model
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Melatih model LSTM
lstm_model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_test_scaled, y_test))

In [ ]:
# Membangun model Autoencoder untuk deteksi Zero-Day Attack
autoencoder = Sequential()
autoencoder.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
autoencoder.add(Dense(32, activation='relu'))
autoencoder.add(Dense(16, activation='relu'))  # Bottleneck layer
autoencoder.add(Dense(32, activation='relu'))
autoencoder.add(Dense(64, activation='relu'))
autoencoder.add(Dense(X_train_scaled.shape[1], activation='sigmoid'))  # Rekonstruksi data

# Kompilasi model
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# Melatih model Autoencoder
autoencoder.fit(X_train_scaled, X_train_scaled, epochs=10, batch_size=32, validation_data=(X_test_scaled, X_test_scaled))

In [ ]:
# Prediksi dari Autoencoder (untuk mendeteksi Zero-Day Attack)
autoencoder_predictions = autoencoder.predict(X_test_scaled)

# Hitung reconstruction error
reconstruction_error = np.mean(np.abs(X_test_scaled - autoencoder_predictions), axis=1)

# Tentukan threshold untuk deteksi Zero-Day Attack
threshold = 0.5  # Ambang batas error, jika lebih tinggi dianggap sebagai Zero-Day Attack

# Identifikasi zero-day attack berdasarkan reconstruction error
zero_day_predictions = reconstruction_error > threshold

# Menampilkan hasil deteksi Zero-Day Attack
for i, prediction in enumerate(zero_day_predictions):
    if prediction:
        print(f"Sampel {i}: Zero-Day Attack terdeteksi!")
    else:
        print(f"Sampel {i}: Normal (tidak ada serangan)")

In [ ]:
# Prediksi dari model LSTM (untuk klasifikasi serangan yang dikenal)
lstm_predictions = lstm_model.predict(X_test_scaled)
lstm_predictions_class = np.argmax(lstm_predictions, axis=1)  # Mengambil kelas dengan probabilitas tertinggi

# Gabungkan hasil dari LSTM dan Autoencoder menggunakan weighted average
final_scores = (0.7 * lstm_predictions_class) + (0.3 * reconstruction_error)

# Tentukan ancaman berdasarkan skor akhir
final_predictions = final_scores > threshold  # Threshold gabungan

# Evaluasi hasil deteksi ancaman
accuracy = accuracy_score(y_test, final_predictions)
precision = precision_score(y_test, final_predictions, average='macro')
recall = recall_score(y_test, final_predictions, average='macro')
f1 = f1_score(y_test, final_predictions, average='macro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# Pemetaan label numerik ke jenis serangan
attack_labels = {
    0: 'normal',
    1: 'DDoS',
    2: 'ransomware',
    3: 'data injection',
    4: 'XSS',
    5: 'MitM',
    6: 'Zero-Day Attack'  # Tambahkan label untuk Zero-Day Attack
}

# Menampilkan jenis serangan berdasarkan hasil prediksi
for i, prediction in enumerate(final_predictions):
    if prediction:
        # Menggunakan prediksi dari LSTM untuk menentukan jenis serangan
        predicted_label = lstm_predictions_class[i]  # Prediksi dari LSTM
        attack_type = attack_labels.get(int(predicted_label), 'Unknown')  # Pemetaan ke jenis serangan
        print(f"Sampel {i}: Terdeteksi serangan: {attack_type}")
    else:
        print(f"Sampel {i}: Normal (tidak ada serangan)")